# Daily Climate Time-Series Forecasting

**Goal:** predict daily mean temperature (`meantemp`) using historical weather patterns.

This notebook practices a complete time-series forecasting workflow:

1. Load and inspect daily climate data
2. Convert the date column into a proper datetime index
3. Visualize trends and seasonality
4. Detect and clean pressure outliers
5. Create lag, rolling-window, and calendar features
6. Avoid data leakage by using only past information
7. Evaluate an XGBoost model using `TimeSeriesSplit`

The main forecasting question is:

> Can we predict today's mean temperature using information available from previous days?

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling and evaluation
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor, plot_importance

# Display settings
pd.set_option("display.max_columns", None)

## 1. Load the data

The dataset is indexed by date. For time-series forecasting, the date must be converted to a real `datetime` index, not left as a string/object index.

In [ ]:
# Load data
# In Colab/Kaggle, update this path if your CSV is stored somewhere else.
data = pd.read_csv(
    "/content/DailyDelhiClimateTrain.csv",
    index_col="date"
)

# Convert index to datetime so pandas can understand time order
# This is needed for time-aware operations such as interpolation, plotting, and date feature extraction.
data.index = pd.to_datetime(data.index)

# Keep rows sorted chronologically
# This is important because forecasting must respect time order.
data = data.sort_index()

display(data.head())
print(data.info())

## 2. Basic data checks

Before modeling, check:

- Missing values
- Duplicate dates
- Whether the index is sorted in chronological order
- Basic descriptive statistics

In [ ]:
print("Missing values:")
print(data.isna().sum())

print("
Index dtype:", data.index.dtype)
print("Chronologically sorted:", data.index.is_monotonic_increasing)
print("Duplicate dates:", data.index.duplicated().sum())

display(data.describe())

## 3. Visualize the original time series

Plotting each column helps us inspect:

- Trend
- Seasonality
- Noise
- Outliers
- Suspicious sensor values

`meantemp` shows strong yearly seasonality. `meanpressure` contains suspicious extreme values that need further inspection.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(data.columns):
    axes[i].plot(data.index, data[col])
    axes[i].set_title(col)
    axes[i].set_xlabel("Date")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. Investigate pressure outliers

Atmospheric pressure should usually be around 1000 hPa. Values such as negative pressure or extremely large pressure are not physically realistic, so they are likely sensor/data-entry errors.

We first inspect the distribution before cleaning.

In [ ]:
print(data["meanpressure"].describe())

plt.figure(figsize=(10, 4))
data["meanpressure"].hist(bins=100)
plt.title("Mean Pressure Distribution Before Cleaning")
plt.xlabel("Mean Pressure")
plt.ylabel("Frequency")
plt.show()

print("Lowest pressure values:")
display(data["meanpressure"].sort_values().head(10))

print("Highest pressure values:")
display(data["meanpressure"].sort_values().tail(10))

## 5. Clean pressure outliers with the IQR method

The IQR method marks values below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR` as outliers.

Here, outliers are converted to `NaN`, then filled using time interpolation.

Why interpolation? Pressure usually changes smoothly over time, so estimating invalid values from neighboring dates is reasonable.

In [ ]:
df = data.copy()

Q1 = df["meanpressure"].quantile(0.25)
Q3 = df["meanpressure"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

pressure_outliers = (df["meanpressure"] < lower) | (df["meanpressure"] > upper)

print("Lower threshold:", lower)
print("Upper threshold:", upper)
print("Number of pressure outliers:", pressure_outliers.sum())

display(df.loc[pressure_outliers, ["meanpressure"]])

# Replace outliers with missing values
df.loc[pressure_outliers, "meanpressure"] = np.nan

print("
Missing values after replacing outliers:")
print(df.isna().sum())

# Fill the artificial missing values using time interpolation
df["meanpressure"] = df["meanpressure"].interpolate(method="time")

print("
Missing values after interpolation:")
print(df.isna().sum())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["meanpressure"])
plt.title("Mean Pressure After Outlier Cleaning")
plt.xlabel("Date")
plt.ylabel("Mean Pressure")
plt.show()

plt.figure(figsize=(10, 4))
df["meanpressure"].hist(bins=100)
plt.title("Mean Pressure Distribution After Cleaning")
plt.xlabel("Mean Pressure")
plt.ylabel("Frequency")
plt.show()

## 6. Correlation analysis

Correlation helps us understand linear relationships between variables.

Important reminder:

- Positive correlation means two variables move in the same direction.
- Negative correlation means they move in opposite directions.
- Strength depends on the absolute value, so `-0.88` is a strong relationship, not a weak one.

However, correlation alone does not decide which features are useful for forecasting. Models can also learn nonlinear effects and interactions.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

## 7. Feature engineering

We create three groups of features:

### 1. Lag features
Lag features give the model historical memory.

Example: `meantemp_lag_1` means yesterday's temperature.

### 2. Rolling features
Rolling features summarize recent behavior.

Example: `meantemp_rolling_mean_7` means the average temperature over the last 7 days.

### 3. Calendar features
Calendar features capture seasonality.

Example: `day_of_year` helps the model learn yearly temperature cycles.

In [ ]:
target_col = "meantemp"

# Target lags: previous temperature values
for lag in [1, 7, 14, 30]:
    df[f"meantemp_lag_{lag}"] = df[target_col].shift(lag)

# Rolling temperature features
# These use recent target history to capture trend and variability.
df["meantemp_rolling_mean_7"] = df[target_col].rolling(window=7).mean()
df["meantemp_rolling_mean_30"] = df[target_col].rolling(window=30).mean()
df["meantemp_rolling_std_7"] = df[target_col].rolling(window=7).std()

# Lagged weather predictors
# These prevent same-day leakage. For example, humidity_lag_1 is yesterday's humidity.
for col in ["humidity", "wind_speed", "meanpressure"]:
    df[f"{col}_lag_1"] = df[col].shift(1)
    df[f"{col}_lag_7"] = df[col].shift(7)

# Calendar features
# These help the model capture weekly/yearly seasonal patterns.
df["day_of_week"] = df.index.dayofweek
df["day_of_year"] = df.index.dayofyear
df["month"] = df.index.month

display(df.head(10))
print(df.columns)

## 8. Prepare modeling data and avoid leakage

For a true forecasting setup, we must avoid using information from the same day as the target.

We therefore remove:

- `meantemp` from features because it is the target
- same-day `humidity`, `wind_speed`, and `meanpressure`

Instead, we keep their lagged versions.

In [ ]:
# Drop rows with NaN values created by lag/rolling features
data_model = df.dropna().copy()

# Features: use only past information and calendar features
X = data_model.drop(columns=["meantemp", "humidity", "wind_speed", "meanpressure"])
y = data_model["meantemp"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("
Feature columns:")
print(X.columns.tolist())

## 9. Time-series cross-validation with XGBoost

Random train/test splitting is not appropriate for forecasting because it can mix future and past observations.

`TimeSeriesSplit` keeps chronological order:

- Train on earlier dates
- Test on later dates
- Expand the training window over folds

This simulates a realistic forecasting scenario.

In [ ]:
xgb_reg = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    objective="reg:squarederror"
)

tscv = TimeSeriesSplit(n_splits=5)

mae_scores = []
rmse_scores = []
r2_scores = []

last_fold = {}

for fold, (train_index, test_index) in enumerate(tscv.split(X), start=1):
    x_train = X.iloc[train_index]
    y_train = y.iloc[train_index]

    x_test = X.iloc[test_index]
    y_test = y.iloc[test_index]

    model = xgb_reg.fit(x_train, y_train)
    y_pred = model.predict(x_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)

    print(f"Fold {fold}")
    print("Train period:", x_train.index.min().date(), "to", x_train.index.max().date())
    print("Test period :", x_test.index.min().date(), "to", x_test.index.max().date())
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-" * 40)

    last_fold = {
        "x_test": x_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "model": model
    }

print("
Average Results")
print("MAE :", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("R2  :", np.mean(r2_scores))

## 10. Visualize the final fold predictions

This plot compares actual vs predicted temperature for the last validation fold.

A line plot is more useful than a scatter plot in forecasting because it preserves time order.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(last_fold["y_test"].index, last_fold["y_test"], label="Actual")
plt.plot(last_fold["y_test"].index, last_fold["y_pred"], label="Predicted")
plt.title("Actual vs Predicted Mean Temperature - Last TimeSeriesSplit Fold")
plt.xlabel("Date")
plt.ylabel("Mean Temperature")
plt.legend()
plt.show()

## 11. Feature importance

Feature importance helps us understand which variables XGBoost used most often to reduce prediction error.

This does not prove causality, but it is useful for model interpretation.

In [ ]:
plt.figure(figsize=(10, 6))
plot_importance(last_fold["model"], max_num_features=15)
plt.title("XGBoost Feature Importance")
plt.show()

## 12. Summary of findings

Main observations:

- `meantemp` has strong yearly seasonality.
- `meanpressure` contained unrealistic outliers, which were cleaned using the IQR method and time interpolation.
- Same-day weather variables were removed from the feature matrix to avoid leakage.
- Lagged weather variables were used instead, so the model only uses past information.
- `TimeSeriesSplit` was used to evaluate the model across multiple chronological folds.
- XGBoost achieved strong performance using lag, rolling-window, and calendar features.

Important learning point:

> Forecasting is not only about choosing a model. Correct feature engineering and leakage prevention are often more important than the algorithm itself.